# M3 online — Persister DiagOps avec SQLAlchemy et Alembic

## Mission

Modéliser les données DiagOps dans une base relationnelle, y charger les tables préparées, puis **faire évoluer le schéma par migration** pour accueillir les mesures capteurs et les importer sans duplication.

Ce notebook sert de fil conducteur et de démonstration. Le code durable va dans `src/db/` et dans les migrations Alembic, pas dans les cellules.

Le moteur par défaut est SQLite. `DIAGOPS_DATABASE_URL` permet d'utiliser PostgreSQL sans modifier le code.

## 0. Environnement

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import inspect, select, text

from src.db.session import build_engine, build_session_factory, database_url

roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPOSITORY_ROOT = next((root for root in roots if (root / 'data_pack' / 'MANIFEST.yaml').is_file()), None)
DATA_DIR = Path(os.environ.get('DIAGOPS_DATA_DIR', REPOSITORY_ROOT / 'data_pack' / '2026-S1'))
REFERENCE_DIR = DATA_DIR / 'reference_runs' / 'm2_for_m3'

engine = build_engine()
print(database_url())

## 1. Justifier le modèle de stockage

Avant de déclarer un modèle, dites pourquoi une base relationnelle convient à ces données, et pourquoi une autre forme conviendrait moins bien.

Les mesures capteurs sont un cas discutable : volumétrie élevée, schéma stable, écritures en lot, lectures par plage de temps et par équipement. Base documentaire, fichiers colonnes ou base orientée séries temporelles sont des alternatives défendables. Les trois tables héritées de M2 ne posent pas la même question.

Comparez **au moins deux modèles** sur des critères explicites — requêtes attendues, contraintes d'intégrité à faire respecter, volumétrie et croissance, coût d'exploitation — puis justifiez celui que vous retenez. Il n'est pas demandé d'installer une seconde base.

In [ ]:
# TODO : quelques mesures qui appuient la comparaison — volumétrie par table,
# taille des fichiers, nombre de séries, requêtes attendues.

## 2. Modèle

`src/db/models.py` déclare `Equipment` comme exemple complet. À écrire :

- `Event` : clé `event_id`, référence `equipment_id`, `start_at` obligatoire, `end_at` facultative ;
- `Maintenance` : clé `maintenance_id`, références `event_id` et `equipment_id` ;
- `SensorReading` : **aucun identifiant de ligne**. La clé logique `equipment_id + timestamp + sensor_name` doit être exprimée par une contrainte d'unicité.

Justifiez vos types : identifiants, dates, montants, catégories, colonnes facultatives.

In [ ]:
from src.db.models import Base, Equipment

sorted(Base.metadata.tables)

### Attention aux clés étrangères sur SQLite

SQLite n'applique pas les clés étrangères par défaut. `build_engine` active `PRAGMA foreign_keys=ON`. Vérifiez-le : une contrainte déclarée mais non appliquée ne protège rien.

In [ ]:
with engine.connect() as connection:
    if engine.dialect.name == 'sqlite':
        print(connection.execute(text('PRAGMA foreign_keys')).scalar())

## 3. Migration initiale

La base est créée **par migration**, pas par `create_all`. Depuis un terminal, à la racine de votre espace de travail :

```bash
alembic init alembic
# dans alembic.ini : sqlalchemy.url
# dans alembic/env.py : target_metadata = src.db.models.Base.metadata
alembic revision --autogenerate -m "schema initial"
alembic upgrade head
```

Relisez la révision générée avant de l'appliquer : l'autogénération propose, elle ne décide pas.

In [ ]:
# Vérification : quelles tables existent réellement après la migration ?
inspect(engine).get_table_names()

## 4. Charger les tables préparées

L'ordre d'insertion est imposé par les clés étrangères. Les lignes refusées sont **comptées et expliquées**, jamais perdues silencieusement.

`src/db/import_sources.py` fournit `import_equipment` comme exemple : insertion ligne à ligne dans un point de sauvegarde, comptage des rejets et de leurs raisons.

In [ ]:
from src.db.import_sources import import_equipment

if 'equipment' not in inspect(engine).get_table_names():
    raise RuntimeError(
        "Table 'equipment' absente : appliquez d'abord la migration initiale "
        "avec `alembic upgrade head`."
    )

session = build_session_factory(engine)()
rapport = import_equipment(session, REFERENCE_DIR / 'processed' / 'equipment.csv')
session.commit()
session.close()
rapport.as_dict()

In [ ]:
# TODO : import des événements puis des interventions, avec le même comptage.
# TODO : expliquer chaque écart entre lignes lues et lignes insérées.

## 5. Migration des mesures

Écrivez une seconde migration qui **ajoute** la table des mesures sur une base déjà chargée.

- contrainte d'unicité exprimant la clé logique ;
- clé étrangère vers `equipment`, et décision sur le sort des mesures orphelines ;
- au moins un index, justifié par une requête que vous exécutez réellement ;
- `downgrade` exécutable : documentez ce qu'il détruit.

```bash
alembic revision --autogenerate -m "ajout des mesures capteurs"
alembic upgrade head
```

Démontrez que les données déjà chargées sont toujours là après la migration.

In [ ]:
# TODO : comptage avant et après migration, sur une table déjà chargée.

## 6. Import idempotent des mesures

`import_measurements` est à écrire. Le contrat est dans sa docstring : deux exécutions successives laissent la table dans le même état.

Plusieurs stratégies conviennent — contrôle préalable des clés existantes, contrainte d'unicité assortie d'une insertion tolérante, insertion avec résolution de conflit. Justifiez le choix et **observez son coût** sur les 50 000 lignes du fichier.

Le fichier reçu contient des mesures dont l'équipement est inconnu et des horodatages hétérogènes. Comptez-les, ne les convertissez pas en silence.

In [ ]:
# TODO : premier import, puis second import. Comparer les comptages et le temps
# d'exécution des deux passages.

## 7. Requêtes de contrôle et d'exploitation

Cinq requêtes au moins, exécutées et conservées. En SQL ou avec l'API SQLAlchemy, au choix.

1. mesures par équipement et par capteur ;
2. première et dernière mesure de chaque série ;
3. équipements sans aucune mesure ;
4. mesures refusées faute d'équipement correspondant ;
5. une requête d'agrégation utile au métier, au choix.

In [ ]:
# TODO : écrire et exécuter les requêtes, puis conserver leurs résultats.

## 8. Index et effet mesuré

Ajoutez un index parce qu'une requête en a besoin, pas par principe. Mesurez le temps de la requête avant et après, et rapportez ce que vous observez.

Sur SQLite, `EXPLAIN QUERY PLAN` indique si l'index est réellement utilisé.

In [ ]:
# TODO : plan d'exécution et temps mesuré, avant et après l'index.

## 9. Apport et coût de la persistance

Concluez : qu'apporte la base par rapport à la lecture directe des CSV, et que coûte-t-elle ? Appuyez-vous sur ce que vous avez mesuré — temps d'import, temps de requête, taille du fichier de base — pas sur une impression.

Le fichier de base de données n'est pas un livrable : il doit pouvoir être reconstruit à partir de votre code.